# CRAFT — End-to-end parameterization walkthrough

This notebook walks through the full CRAFT pipeline using trimethyllysine (KM3) as a worked example.

**What CRAFT produces:**
- `.prepin` — residue topology file for LEaP
- `_gaff.frcmod` — missing GAFF parameters
- `_ff14SB.frcmod` (or `_ff19SB.frcmod`) — missing FF-specific parameters

**Pipeline overview:**

```
Phase 1 (local)   craft-run        cap termini, generate Gaussian + RESP inputs
Phase 2a (HPC)    g16              B3LYP/6-31G* geometry optimisation
Phase 2b (local)  craft-hf-input   extract optimised geometry, write HF input
Phase 2c (HPC)    g16              HF/6-31G(d) single-point for ESP
Phase 3 (local)   craft-amber      espgen → resp → antechamber → prepgen → parmchk2
```

> **Note on Phase 2:** Gaussian runs on an HPC cluster and cannot be executed in this notebook.
> Phases 1 and 3 run locally. For Phase 3, pre-computed Gaussian logs from the KME3 example
> are used so you can follow the full pipeline without submitting jobs.

## 0. Prerequisites

Before running this notebook make sure:

1. CRAFT is installed in the active environment:
   ```bash
   pip install .
   ```
2. AmberTools is installed and the `craft-check` command passes.
3. You are running the notebook from the **CRAFT repo root** (the directory containing `KME3/`).

### Optional: 3D visualization

Molecular visualization uses **py3Dmol**, which is not required to run the pipeline but enhances the walkthrough. Install it with:

```bash
pip install py3Dmol
```

If py3Dmol is not available the visualization cells will print a skip message and the rest of the notebook will continue normally.

In [100]:
import subprocess, sys, shutil
from pathlib import Path

# Verify working directory
assert Path('KME3/KME3.pdb').exists(), \
    'Run this notebook from the CRAFT repo root (the directory containing KME3/)'

print('Working directory OK')

Working directory OK


In [101]:
# Check that CRAFT and AmberTools are available
result = subprocess.run(['craft-check'], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)

Python packages
----------------------------------------
  [ok]      numpy
  [ok]      pyyaml
  [ok]      rdkit  (full symmetry-based RESP equivalence)

Gaussian
----------------------------------------
  [MISSING] g16 or g09 -- not found in $PATH

AmberTools
----------------------------------------
  [ok]      espgen -> /Users/xv24323/miniconda3/envs/AmberTools25/bin/espgen
  [ok]      resp -> /Users/xv24323/miniconda3/envs/AmberTools25/bin/resp
  [ok]      antechamber -> /Users/xv24323/miniconda3/envs/AmberTools25/bin/antechamber
  [ok]      prepgen -> /Users/xv24323/miniconda3/envs/AmberTools25/bin/prepgen
  [ok]      parmchk2 -> /Users/xv24323/miniconda3/envs/AmberTools25/bin/parmchk2

  $AMBERHOME = /Users/xv24323/miniconda3/envs/AmberTools25
  [ok]      parm10.dat found

One or more required tools are missing. See above.




In [102]:
# -- Visualization setup -------------------------------------------------------
try:
    import py3Dmol
    HAS_PY3DMOL = True
    print('py3Dmol available — visualization enabled.')
except ImportError:
    HAS_PY3DMOL = False
    print('py3Dmol not installed.  Install with: pip install py3Dmol')
    print('Visualization cells will print a skip message and continue.')


def _pdb_with_bfactor(pdb_path, values):
    """Return PDB string with values written into the B-factor column."""
    lines_out = []
    val_iter = iter(values)
    with open(pdb_path) as f:
        for line in f:
            if line.startswith(('ATOM', 'HETATM')):
                v = next(val_iter, 0.0)
                line = f"{line[:60]}{v:6.2f}{line[66:].rstrip()}\n"
            lines_out.append(line)
    return ''.join(lines_out)


def _charge_to_hex(charge, max_val):
    """Map a charge value to a red/white/blue hex colour string."""
    t = max(min(charge / max_val, 1.0), -1.0)
    if t < 0:    # red (negative) → white (zero)
        r, g, b = 255, int(255 * (1 + t)), int(255 * (1 + t))
    else:         # white (zero) → blue (positive)
        r, g, b = int(255 * (1 - t)), int(255 * (1 - t)), 255
    return f'#{r:02x}{g:02x}{b:02x}'


def show_pdb(pdb_path, charges=None, label_charges=False, width=620, height=440,
             sphere_radius=0.3, background='0xffffff'):
    """Display a PDB file with py3Dmol."""
    if not HAS_PY3DMOL:
        print('(visualization skipped — install py3Dmol with: pip install py3Dmol)')
        return
    if charges is not None:
        pdb_str = _pdb_with_bfactor(pdb_path, charges)
    else:
        with open(pdb_path) as f:
            pdb_str = f.read()
    view = py3Dmol.view(width=width, height=height)
    view.addModel(pdb_str, 'pdb')
    view.setBackgroundColor(background)
    if charges is not None:
        lim = max(abs(c) for c in charges) or 1.0
        cs  = {'prop': 'b', 'gradient': 'rwb', 'min': -lim, 'max': lim}
        view.setStyle({}, {'stick': {'colorscheme': cs, 'radius': 0.08},
                           'sphere': {'radius': sphere_radius, 'colorscheme': cs}})
        if label_charges:
            from craft.cap import parse_pdb as _parse_pdb
            for atom, chg in zip(_parse_pdb(pdb_path), charges):
                view.addLabel(f"{atom['name']} {chg:+.3f}",
                              {'position': {'x': float(atom['x']),
                                            'y': float(atom['y']),
                                            'z': float(atom['z'])},
                               'fontSize': 10, 'fontColor': 'white',
                               'backgroundColor': 'black', 'backgroundOpacity': 0.6,
                               'showBackground': True})
    else:
        view.setStyle({}, {'stick': {'radius': 0.08}, 'sphere': {'radius': sphere_radius}})
    view.zoomTo()
    return view.show()


def show_xyz(atoms_xyz, charges=None, label_charges=False, width=620, height=440,
             sphere_radius=0.3, background='0xffffff'):
    """
    Display a list of (elem, x, y, z) tuples as returned by parse_opt_log.

    Always loads as XYZ format so hydrogen atoms are never silently dropped
    (py3Dmol filters H atoms when reading PDB format, mirroring the convention
    that X-ray structures omit them).  Charge colouring is applied per-atom
    via index selectors rather than through the B-factor column.

    charges       : list of floats aligned with atoms_xyz — colours by charge.
    label_charges : annotate each atom with elem + charge value (requires charges).
    """
    if not HAS_PY3DMOL:
        print('(visualization skipped — install py3Dmol with: pip install py3Dmol)')
        return

    # Build XYZ string — all atoms (including H) are always preserved.
    lines = [str(len(atoms_xyz)), 'geometry']
    for elem, x, y, z in atoms_xyz:
        lines.append(f'{elem}  {x:.6f}  {y:.6f}  {z:.6f}')

    view = py3Dmol.view(width=width, height=height)
    view.addModel('\n'.join(lines), 'xyz')
    view.setBackgroundColor(background)

    if charges is not None:
        lim = max(abs(c) for c in charges) or 1.0
        # Apply a uniform base style first, then override colour per atom.
        view.setStyle({}, {'stick': {'radius': 0.08}, 'sphere': {'radius': sphere_radius}})
        for idx, ((elem, x, y, z), chg) in enumerate(zip(atoms_xyz, charges)):
            color = _charge_to_hex(chg, lim)
            view.setStyle({'index': idx},
                          {'stick': {'color': color, 'radius': 0.08},
                           'sphere': {'color': color, 'radius': sphere_radius}})
        if label_charges:
            for (elem, x, y, z), chg in zip(atoms_xyz, charges):
                view.addLabel(f"{elem} {chg:+.3f}",
                              {'position': {'x': float(x), 'y': float(y), 'z': float(z)},
                               'fontSize': 10, 'fontColor': 'black',
                               'backgroundColor': 'white', 'backgroundOpacity': 0.7,
                               'showBackground': True})
    else:
        view.setStyle({}, {'stick': {'radius': 0.08}, 'sphere': {'radius': sphere_radius}})

    view.zoomTo()
    return view.show()

py3Dmol available — visualization enabled.


## 1. Input PDB

CRAFT takes a single-residue PDB as input — a free amino acid, a residue cut from a crystal structure, or anything in between. CRAFT inspects the termini automatically and handles all common protonation states.

The KM3 residue is trimethyllysine: lysine with three methyl groups on the ε-amino nitrogen, giving a net charge of +2.

In [103]:
# Input PDB — raw uncapped residue
show_pdb('KME3/KME3.pdb')

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [104]:
with open('KME3/KME3.pdb') as f:
    print(f.read())

TITLE     Structure1
ATOM      1  N   KM3 A  17      21.362   4.712   5.690  1.00  0.00           N1+
ATOM      2  CA  KM3 A  17      21.611   5.728   4.679  1.00  0.00           C  
ATOM      3  C   KM3 A  17      22.926   5.413   3.952  1.00  0.00           C  
ATOM      4  O   KM3 A  17      23.598   4.426   4.253  1.00  0.00           O  
ATOM      5  CB  KM3 A  17      21.608   7.103   5.352  1.00  0.00           C  
ATOM      6  CG  KM3 A  17      21.681   8.266   4.357  1.00  0.00           C  
ATOM      7  CD  KM3 A  17      21.731   9.655   5.005  1.00  0.00           C  
ATOM      8  CE  KM3 A  17      21.805  10.833   4.024  1.00  0.00           C  
ATOM      9  NZ  KM3 A  17      21.851  12.142   4.698  1.00  0.00           N1+
ATOM     10  C1  KM3 A  17      21.922  13.247   3.656  1.00  0.00           C  
ATOM     11  C2  KM3 A  17      20.600  12.323   5.543  1.00  0.00           C  
ATOM     12  C3  KM3 A  17      23.079  12.210   5.592  1.00  0.00           C  
ATOM   

## 2. Configuration

All pipeline settings live in `config.yaml`. Key fields:

| Field | Description |
|---|---|
| `residue.input_pdb` | Path to the raw single-residue PDB |
| `residue.charge` | Net molecular charge |
| `residue.position` | `middle` \| `cterm` \| `nterm` |
| `amber.forcefield` | `ff14SB` \| `ff19SB` \| `~` to skip |

We will parameterize KM3 in the `middle` position (ACE-capped N-terminus, NME-capped C-terminus), which is the most common case for a residue embedded in a peptide chain.

In [105]:
with open('KME3/config_middle.yaml') as f:
    print(f.read())

# CRAFT configuration -- KM3 trimethyllysine, middle position
#
# Run from this directory (KME3/).  Outputs land in KM3/middle/.
#
# -- If CRAFT is installed as a package (pip install .) ----------------------
#
#   Manually (step-by-step):
#     craft-run                                         # Phase 1
#     craft-hf-input KM3/middle/KM3_opt.log            # Phase 2b
#     craft-amber    KM3/middle/KM3_hf.log             # Phase 3
#
#   Single SLURM job (full pipeline):
#     craft-slurm                          # generates KM3/middle/KM3_craft.sh
#     cd KM3/middle && sbatch KM3_craft.sh
#
# -- If running directly from source (no install) ----------------------------
#
#   Manually (step-by-step):
#     python run.py                                     # Phase 1
#     python make_hf_input.py KM3/middle/KM3_opt.log   # Phase 2b
#     python amber_pipeline.py KM3/middle/KM3_hf.log   # Phase 3
#
#   Single SLURM job (full pipeline):
#     python make_slurm.py                 # genera

## 3. Phase 1 — Cap termini and generate all pre-Gaussian inputs

`craft-run` does the following:

1. **Caps the termini** with ACE and NME (or just one, depending on position)
2. **Writes the Gaussian geometry optimisation input** (`_opt.com`)
3. **Writes RESP input files** (`resp.in`, `resp.qin`) with backbone atoms fixed to ff14SB/ff19SB values
4. **Writes the prepgen main-chain file** (`.mc`)

All outputs land in `KM3/middle/`.

In [120]:
result = subprocess.run(
    ['craft-run', '--config', 'config_middle.yaml'],
    capture_output=True, text=True, cwd='KME3'
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)
    sys.exit(result.returncode)

Step 1 -- Cap termini  [middle]
Input : KME3.pdb  (34 atoms, residue KM3)
  N-terminus : NH3+ -- zwitterionic free amino acid
  C-terminus : COO- -- has O1 (1 extra O), free amino acid C-terminus
  Position   : middle
  Action     : remove 3 H(s) on N, add amide-H, remove O1, add ACE + NME
Output: KM3/middle/KM3_capped.pdb
  ACE  resSeq=1 :  6 atoms
  KM3  resSeq=2 : 31 atoms
  NME  resSeq=3 :  6 atoms
  Total         : 43 atoms

Step 2 -- Geometry-optimisation Gaussian input
Input  : KM3/middle/KM3_capped.pdb  (43 atoms)
Output : KM3/middle/KM3_opt.com
  Charge / mult : 1 / 1
  nproc / mem   : 16 / 512MB

Steps 3-5 -- RESP and prepgen inputs
  resp.in  : KM3/middle/resp.in  (43 atoms, 25 sidechain free)
  resp.qin : KM3/middle/resp.qin  (43 charges, 25 sidechain free)
  KM3.mc          : HEAD=N CA=CA TAIL=C omit 12 cap atoms

All Phase 1 outputs written to: KM3/middle/

Next steps:
  1. Submit KM3/middle/KM3_opt.com to HPC
  2. Copy KM3_opt.log into KM3/middle/, then:
       craft-hf-

In [121]:
# Capped PDB — ACE (left) + KM3 (centre) + NME (right)
show_pdb('KME3/KM3/middle/KM3_capped.pdb')

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [122]:
# Inspect the capped PDB — ACE + KM3 + NME
with open('KME3/KM3/middle/KM3_capped.pdb') as f:
    print(f.read())

ATOM      1  CH3 ACE     1      20.042   3.684   7.511  1.00  0.00           C
ATOM      2  H11 ACE     1      21.002   3.315   7.871  1.00  0.00           H
ATOM      3  H21 ACE     1      19.479   2.863   7.069  1.00  0.00           H
ATOM      4  H31 ACE     1      19.479   4.104   8.345  1.00  0.00           H
ATOM      5  C4  ACE     1      20.274   4.762   6.462  1.00  0.00           C
ATOM      6  O1  ACE     1      19.459   5.673   6.332  1.00  0.00           O
ATOM      7  N   KM3     2      21.362   4.712   5.690  1.00  0.00           N
ATOM      8  H   KM3     2      22.007   3.947   5.829  1.00  0.00           H
ATOM      9  CA  KM3     2      21.611   5.728   4.679  1.00  0.00           C
ATOM     10  C   KM3     2      22.926   5.413   3.952  1.00  0.00           C
ATOM     11  O   KM3     2      23.598   4.426   4.253  1.00  0.00           O
ATOM     12  CB  KM3     2      21.608   7.103   5.352  1.00  0.00           C
ATOM     13  CG  KM3     2      21.681   8.266   4.3

In [125]:
# Inspect resp.qin — fixed backbone charges, free sidechain atoms (charge = 0.0)
# with open('KME3/KM3/middle/resp.in') as f:
#     print(f.read())

In [123]:
# Inspect resp.qin — fixed backbone charges, free sidechain atoms (charge = 0.0)
with open('KME3/KM3/middle/resp.qin') as f:
    print(f.read())

 -0.366200  0.112300  0.112300  0.112300  0.597200 -0.567900 -0.415700  0.271900
  0.033700  0.597200 -0.567900  0.000000  0.000000  0.000000  0.000000  0.000000
  0.000000  0.000000  0.000000  0.080800  0.000000  0.000000  0.000000  0.000000
  0.000000  0.000000  0.000000  0.000000  0.000000  0.000000  0.000000  0.000000
  0.000000  0.000000  0.000000  0.000000  0.000000 -0.415700  0.271900 -0.149000
  0.097600  0.097600  0.097600



In [110]:
# Optimised geometry extracted from the Gaussian opt log
from craft.gaussian import parse_opt_log
opt_geom = parse_opt_log('KME3/KM3/middle/KM3_opt.log')
show_xyz(opt_geom)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

### RESP charge constraints

CRAFT fixes the six backbone atoms to standard ff14SB/ff19SB values:

| Atom | Charge |
|------|--------|
| N    | −0.4157 |
| H    | +0.2719 |
| CA   | +0.0337 |
| HA   | +0.0823 |
| C    | +0.5973 |
| O    | −0.5679 |

The HA charge is adjusted by −0.0015 so that the six backbone atoms sum to exactly zero, preventing charge transfer artifacts when the residue is inserted into a protein. Sidechain atoms are free (shown as `0.0` in `resp.qin`).

## 4. Phase 2 — Gaussian calculations (HPC)

Phase 2 requires Gaussian 16 and runs on an HPC cluster. CRAFT generates the necessary input files; you submit them and copy the logs back.

**Phase 2a — Geometry optimisation (B3LYP/6-31G*)**
```bash
# On the HPC cluster:
g16 < KM3/middle/KM3_opt.com > KM3/middle/KM3_opt.log
```

**Phase 2b — Build HF input from optimised geometry**
```bash
craft-hf-input KM3/middle/KM3_opt.log --config config_middle.yaml
```

**Phase 2c — HF/6-31G(d) single-point for ESP**
```bash
# On the HPC cluster:
g16 < KM3/middle/KM3_hf.com > KM3/middle/KM3_hf.log
```

Alternatively, `craft-slurm` generates a single SLURM script that runs the entire pipeline automatically:
```bash
craft-slurm --config config_middle.yaml
cd KM3/middle && sbatch KM3_craft.sh
```

The cell below inspects the Gaussian opt input that Phase 1 generated:

In [111]:
with open('KME3/KM3/middle/KM3_opt.com') as f:
    print(f.read())

%nprocshared=16
%mem=512MB
#P b3lyp/6-31g* opt

KM3  b3lyp/6-31g* geometry optimisation

1 1
 C        20.04200000      3.68400000      7.51100000
 H        21.00200000      3.31500000      7.87100000
 H        19.47900000      2.86300000      7.06900000
 H        19.47900000      4.10400000      8.34500000
 C        20.27400000      4.76200000      6.46200000
 O        19.45900000      5.67300000      6.33200000
 N        21.36200000      4.71200000      5.69000000
 H        22.00700000      3.94700000      5.82900000
 C        21.61100000      5.72800000      4.67900000
 C        22.92600000      5.41300000      3.95200000
 O        23.59800000      4.42600000      4.25300000
 C        21.60800000      7.10300000      5.35200000
 C        21.68100000      8.26600000      4.35700000
 C        21.73100000      9.65500000      5.00500000
 C        21.80500000     10.83300000      4.02400000
 N        21.85100000     12.14200000      4.69800000
 C        21.92200000     13.24700000      

For the rest of this notebook we use the **pre-computed Gaussian logs** included in the KME3 example, so you can follow Phase 3 without submitting jobs.

In [112]:
# Structure coloured by RESP charge — red = negative, white = zero, blue = positive
show_pdb('KME3/KM3/middle/KM3_capped.pdb', charges=charges)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [113]:
result = subprocess.run(
    ['craft-amber', 'KM3/middle/KM3_hf.log', '--config', 'config_middle.yaml'],
    capture_output=True, text=True, cwd='KME3'
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)
    sys.exit(result.returncode)

Residue  : KM3  (charge +1)
Position : middle
Log      : KM3/middle/KM3_hf.log
MC file  : /Users/xv24323/Documents/github/CRAFT/KME3/KM3/middle/KM3.mc


-- espgen ------------------------------------------------------------
  $ espgen -i /Users/xv24323/Documents/github/CRAFT/KME3/KM3/middle/KM3_hf.log -o esp.dat


-- resp --------------------------------------------------------------
  $ resp -O -i resp.in -o resp.out -p resp.pch -t resp.chg -q resp.qin -e esp.dat

-- antechamber -------------------------------------------------------
  $ antechamber -fi gout -i /Users/xv24323/Documents/github/CRAFT/KME3/KM3/middle/KM3_hf.log -bk KM3 -fo ac -o KM3.ac -c rc -cf resp.chg -at amber -nc 1
Info: acdoctor mode is on: check and diagnose problems in the input file.
Info: The atom type is set to amber; the options available to the -at flag are
      gaff, gaff2, amber, bcc, abcg2, and sybyl.

-- Check Format for Gaussian Output File --
   Status: pass

-- remap atom names ----------------------

### Fitted charges

After `resp` runs, `resp.chg` contains the fitted partial charges. Backbone atoms will match their fixed values exactly; sidechain charges are the RESP-fitted values.

In [114]:
# Show RESP-fitted charges alongside atom names from the capped PDB
from craft.cap import parse_pdb

atoms   = parse_pdb('KME3/KM3/middle/KM3_capped.pdb')
charges = [float(x) for x in open('KME3/KM3/middle/resp.chg').read().split()]

print(f"{'Atom':<6} {'Res':<6} {'Charge':>8}")
print('-' * 24)
for atom, charge in zip(atoms, charges):
    print(f"{atom['name']:<6} {atom['resName']:<6} {charge:>8.4f}")

Atom   Res      Charge
------------------------
CH3    ACE     -0.3662
H11    ACE      0.1123
H21    ACE      0.1123
H31    ACE      0.1123
C4     ACE      0.5972
O1     ACE     -0.5679
N      KM3     -0.4157
H      KM3      0.2719
CA     KM3      0.0337
C      KM3      0.5972
O      KM3     -0.5679
CB     KM3      0.0057
CG     KM3     -0.0499
CD     KM3      0.0282
CE     KM3     -0.0614
NZ     KM3      0.0612
C1     KM3     -0.3356
C2     KM3     -0.3356
C3     KM3     -0.3356
HA     KM3      0.0808
HB2    KM3      0.0260
HB3    KM3      0.0260
HG2    KM3      0.0354
HG3    KM3      0.0354
HD2    KM3      0.0262
HD3    KM3      0.0262
HE2    KM3      0.1074
HE3    KM3      0.1074
H1     KM3      0.1814
H2     KM3      0.1814
H3     KM3      0.1814
H4     KM3      0.1814
H5     KM3      0.1814
H6     KM3      0.1814
H7     KM3      0.1814
H8     KM3      0.1814
H9     KM3      0.1814
N1     NME     -0.4157
H10    NME      0.2719
C5     NME     -0.1490
H12    NME      0.0976
H22    NM

### Output files

In [115]:
# Residue topology
with open('KME3/KM3/middle/KM3.prepin') as f:
    print(f.read())

    0    0    2

This is a remark line
molecule.res
KM3   INT  0
CORRECT     OMIT DU   BEG
  0.0000
   1  DUMM  DU    M    0  -1  -2     0.000      .0        .0      .00000
   2  DUMM  DU    M    1   0  -1     1.449      .0        .0      .00000
   3  DUMM  DU    M    2   1   0     1.523   111.21       .0      .00000
   4  N     N     M    3   2   1     1.540   111.208  -180.000 -0.415700
   5  H     H     E    4   3   2     1.013   110.699  -103.053  0.271900
   6  CA    CT    M    4   3   2     1.460    39.946   147.684  0.033700
   7  CB    CT    3    6   4   3     1.542   112.932    13.110  0.005726
   8  CG    CT    3    7   6   4     1.535   112.695  -159.897 -0.049889
   9  CD    CT    3    8   7   6     1.541   112.166  -178.363  0.028173
  10  CE    CT    3    9   8   7     1.526   109.143   179.181 -0.061375
  11  NZ    N3    3   10   9   8     1.538   116.671  -178.470  0.061208
  12  C1    CT    3   11  10   9     1.508   108.002  -179.658 -0.335569
  13  H4    HP    E   12

In [119]:
# Same view with per-atom charge labels (zoom in to read individual values)
show_xyz(opt_geom, label_charges=True, charges=charges, width=1000, height=650)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [117]:
# Optimised geometry coloured by RESP-fitted charges
# red = negative  /  white = zero  /  blue = positive
show_xyz(opt_geom, charges=charges)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [118]:
# ff14SB missing parameters
with open('KME3/KM3/middle/KM3_ff14SB.frcmod') as f:
    print(f.read())

Remark line goes here
MASS
CT 12.010        0.878
HC 1.008         0.135
C  12.010        0.616
O  16.000        0.434
N  14.010        0.530
H  1.008         0.161
N3 14.010        0.530
H1 1.008         0.135
HP 1.008         0.135

BOND
CT-HC  340.00   1.090
C -CT  317.00   1.522
C -O   570.00   1.229
C -N   490.00   1.335
H -N   434.00   1.010
CT-N   337.00   1.449
CT-CT  310.00   1.526
CT-H1  340.00   1.090
CT-N3  367.00   1.471
CT-HP  340.00   1.090

ANGLE
CT-C -O    80.000     120.400
CT-C -N    70.000     116.600
HC-CT-HC   35.000     109.500
C -CT-HC   50.000     109.500
C -N -H    50.000     120.000
C -N -CT   50.000     121.900
N -C -O    80.000     122.900
C -CT-N    63.000     110.100
CT-CT-N    80.000     109.700
H1-CT-N    50.000     109.500
CT-N -H    50.000     118.040
CT-CT-CT   40.000     109.500
CT-CT-HC   50.000     109.500
C -CT-CT   63.000     111.100
C -CT-H1   50.000     109.500
CT-CT-H1   50.000     109.500
CT-CT-N3   80.000     111.200
CT-CT-HP   50.000     1

## 6. Using the outputs in LEaP

The `.prepin` and `.frcmod` files are loaded into LEaP alongside the standard force field to build a system containing the modified residue.

```
# tleap input example
source leaprc.protein.ff14SB

loadAmberPrep  KM3/middle/KM3.prepin
loadAmberParams KM3/middle/KM3_ff14SB.frcmod

mol = sequence { ACE KM3 NME }
saveAmberParm mol system.prmtop system.inpcrd
quit
```

For a full protein containing this residue, load the PDB after the `loadAmberPrep` and `loadAmberParams` calls — LEaP will recognise KM3 as a known residue.

## 7. All three terminal positions

A complete parameterization requires three variants. The KME3 folder includes configs for all three:

In [ ]:
positions = [
    ('middle', 'KM3/middle/KM3_hf.log',      'config_middle.yaml'),
    ('cterm',  'KM3/cterm/KM3_cterm_hf.log', 'config_cterm.yaml'),
    ('nterm',  'KM3/nterm/KM3_nterm_hf.log', 'config_nterm.yaml'),
]

for position, hf_log, config in positions:
    sep = '=' * 60
    print(f'\n{sep}')
    print(f'Position: {position}')
    print(sep)
    result = subprocess.run(
        ['craft-amber', hf_log, '--config', config],
        capture_output=True, text=True, cwd='KME3'
    )
    print(result.stdout)
    if result.returncode != 0:
        print('STDERR:', result.stderr)

In [ ]:
# Summary of output files for all three positions
output_files = [
    'KME3/KM3/middle/KM3.prepin',
    'KME3/KM3/middle/KM3_ff14SB.frcmod',
    'KME3/KM3/cterm/KM3_cterm.prepin',
    'KME3/KM3/cterm/KM3_cterm_ff14SB.frcmod',
    'KME3/KM3/nterm/KM3_nterm.prepin',
    'KME3/KM3/nterm/KM3_nterm_ff14SB.frcmod',
]

print(f"{'File':<55} {'Status'}")
print('-' * 62)
for f in output_files:
    status = 'OK' if Path(f).exists() else 'MISSING'
    print(f"{f:<55} {status}")